<a href="https://colab.research.google.com/github/Aritra2004679/HyBert-X/blob/main/notebooks/Finetuned_hb_and_cb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# Step 1: Mount Google Drive
# ==========================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================
# Step 2: Load Dataset
# ==========================================

import pandas as pd
import os

PROJECT_PATH = "/content/drive/MyDrive/Cyberbullying_Project"

df = pd.read_csv(os.path.join(PROJECT_PATH, "cyberbullying.csv"))

print("✅ Dataset Loaded Successfully!")
print("Dataset Shape:", df.shape)
df.head()

✅ Dataset Loaded Successfully!
Dataset Shape: (684383, 14)


,text,religious_hate,ethnic_hate,age_discrimination,gender_hate,sexual_harassment,threats,body_shaming,political_hate,trolling,mental_hate,discrimination,other_cyberbullying_types,not_cyberbullying
0,Lol you are an idiot for believing in those la...,1,0,0,0,0,0,0,0,0,0,0,0,0
1,Bro Im Christian but Im not Christian-Christia...,1,0,0,0,0,0,0,0,0,0,0,0,0
2,I dont like religion I just like the music,1,0,0,0,0,0,0,0,0,0,0,0,0
3,I hate religion its a brainwashing cult,1,0,0,0,0,0,0,0,0,0,0,0,0
4,You know what I have heard enough of your piou...,1,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
# ==========================================
# Step 3: Define Common Project Paths
# ==========================================

import os

PROJECT_PATH = "/content/drive/MyDrive/Cyberbullying_Project"

DATASET_PATH = os.path.join(PROJECT_PATH, "cyberbullying.csv")

print("Project Directory :", PROJECT_PATH)
print("Dataset Path      :", DATASET_PATH)

Project Directory : /content/drive/MyDrive/Cyberbullying_Project
Dataset Path      : /content/drive/MyDrive/Cyberbullying_Project/cyberbullying.csv


In [ ]:
# ==========================================
# Step 4: Install Required Libraries
# ==========================================

!pip install transformers datasets accelerate evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00


In [ ]:
# ==========================================
# Step 5: Import Required Libraries
# ==========================================

import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
# ==========================================
# Step 6: Inspect Dataset
# ==========================================

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
df.head()

Dataset Shape: (684383, 14)

Column Names:
['text', 'religious_hate', 'ethnic_hate', 'age_discrimination', 'gender_hate', 'sexual_harassment', 'threats', 'body_shaming', 'political_hate', 'trolling', 'mental_hate', 'discrimination', 'other_cyberbullying_types', 'not_cyberbullying']

First 5 rows:


,text,religious_hate,ethnic_hate,age_discrimination,gender_hate,sexual_harassment,threats,body_shaming,political_hate,trolling,mental_hate,discrimination,other_cyberbullying_types,not_cyberbullying
0,Lol you are an idiot for believing in those la...,1,0,0,0,0,0,0,0,0,0,0,0,0
1,Bro Im Christian but Im not Christian-Christia...,1,0,0,0,0,0,0,0,0,0,0,0,0
2,I dont like religion I just like the music,1,0,0,0,0,0,0,0,0,0,0,0,0
3,I hate religion its a brainwashing cult,1,0,0,0,0,0,0,0,0,0,0,0,0
4,You know what I have heard enough of your piou...,1,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
# ==========================================
# Step 7: Define Text and Label Columns
# ==========================================

TEXT_COLUMN = "text"

# All columns except the text column are treated as labels
label_columns = [col for col in df.columns if col != TEXT_COLUMN]

NUM_LABELS = len(label_columns)

print("Text Column:", TEXT_COLUMN)
print("Number of Labels:", NUM_LABELS)

print("\nLabel Columns:")
for i, label in enumerate(label_columns, start=1):
    print(f"{i}. {label}")

Text Column: text
Number of Labels: 13

Label Columns:
1. religious_hate
2. ethnic_hate
3. age_discrimination
4. gender_hate
5. sexual_harassment
6. threats
7. body_shaming
8. political_hate
9. trolling
10. mental_hate
11. discrimination
12. other_cyberbullying_types
13. not_cyberbullying


In [ ]:
# ==========================================
# Step 8: Create Train-Validation Split
# ==========================================

from sklearn.model_selection import train_test_split
import numpy as np

# Features (text)
X = df[TEXT_COLUMN]

# Multi-label targets
y = df[label_columns].values

# Train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

# =====================================================
# Select a random subset of the training data
# =====================================================

TRAIN_SIZE = 200000

np.random.seed(42)

indices = np.random.choice(
    len(X_train),
    size=TRAIN_SIZE,
    replace=False
)

X_train = X_train.iloc[indices].reset_index(drop=True)
y_train = y_train[indices]

print("=" * 50)
print("Training samples :", len(X_train))
print("Validation samples:", len(X_val))
print("Training subset used for fine-tuning :", TRAIN_SIZE)
print("=" * 50)

Training samples : 200000
Validation samples: 68439
Training subset used for fine-tuning : 200000


In [ ]:
# ==========================================
# HateBERT: Load Tokenizer
# ==========================================

HATEBERT_MODEL_NAME = "GroNLP/hateBERT"

HATEBERT_TOKENIZER = AutoTokenizer.from_pretrained(HATEBERT_MODEL_NAME)

print("HateBERT tokenizer loaded successfully!")
print("Vocabulary size:", HATEBERT_TOKENIZER.vocab_size)

config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

HateBERT tokenizer loaded successfully!
Vocabulary size: 30522


In [ ]:
# ==========================================
# CyberBERT: Load Tokenizer
# ==========================================

CYBERBERT_MODEL_NAME = "bert-base-uncased"

CYBERBERT_TOKENIZER = AutoTokenizer.from_pretrained(CYBERBERT_MODEL_NAME)

print("CyberBERT tokenizer loaded successfully!")
print("Vocabulary size:", CYBERBERT_TOKENIZER.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

CyberBERT tokenizer loaded successfully!
Vocabulary size: 30522


In [ ]:
# ==========================================
# Step 10: Define Cyberbullying Dataset
# ==========================================

from torch.utils.data import Dataset

class CyberbullyingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(
                self.labels[idx],
                dtype=torch.float
            )
        }

In [ ]:
# ==========================================
# HateBERT: Create Datasets
# ==========================================

hatebert_train_dataset = CyberbullyingDataset(
    X_train,
    y_train,
    HATEBERT_TOKENIZER
)

hatebert_val_dataset = CyberbullyingDataset(
    X_val,
    y_val,
    HATEBERT_TOKENIZER
)

print("HateBERT training dataset size:", len(hatebert_train_dataset))
print("HateBERT validation dataset size:", len(hatebert_val_dataset))

HateBERT training dataset size: 200000
HateBERT validation dataset size: 68439


In [ ]:
# ==========================================
# CyberBERT: Create Datasets
# ==========================================

cyberbert_train_dataset = CyberbullyingDataset(
    X_train,
    y_train,
    CYBERBERT_TOKENIZER
)

cyberbert_val_dataset = CyberbullyingDataset(
    X_val,
    y_val,
    CYBERBERT_TOKENIZER
)

print("CyberBERT training dataset size:", len(cyberbert_train_dataset))
print("CyberBERT validation dataset size:", len(cyberbert_val_dataset))

CyberBERT training dataset size: 200000
CyberBERT validation dataset size: 68439


In [ ]:
# ==========================================
# HateBERT: Load Base Model
# ==========================================

from transformers import AutoModel

HATEBERT_BASE_MODEL = AutoModel.from_pretrained(
    HATEBERT_MODEL_NAME
)

print("HateBERT base model downloaded successfully!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HateBERT base model downloaded successfully!


In [ ]:
# ==========================================
# CyberBERT: Load Base Model
# ==========================================

CYBERBERT_BASE_MODEL = AutoModel.from_pretrained(
    CYBERBERT_MODEL_NAME
)

print("✅ CyberBERT base model downloaded successfully!")

# Save the base model to Google Drive
CYBERBERT_BASE_MODEL.save_pretrained(
    CYBERBERT_MODEL_PATH
)

print("✅ CyberBERT base model saved to:")
print(CYBERBERT_MODEL_PATH)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ CyberBERT base model downloaded successfully!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ CyberBERT base model saved to:
/content/drive/MyDrive/Cyberbullying_Project/CyberBERT_Pretrained


In [ ]:
# ==========================================
# Step 13A: Load HateBERT for Fine-Tuning
# ==========================================

from transformers import AutoModelForSequenceClassification

HATEBERT_MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "HateBERT_Pretrained"
)

hatebert_model = AutoModelForSequenceClassification.from_pretrained(
    HATEBERT_MODEL_PATH,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)

print("✅ HateBERT loaded for fine-tuning.")
print("Number of labels:", hatebert_model.config.num_labels)
print("Problem Type:", hatebert_model.config.problem_type)

OSError: Error no file named model.safetensors, or pytorch_model.bin, found in directory /content/drive/MyDrive/Cyberbullying_Project/HateBERT_Pretrained.

In [ ]:
# ==========================================
# Step 13B: Load CyberBERT for Fine-Tuning
# ==========================================

CYBERBERT_MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "CyberBERT_Pretrained"
)

cyberbert_model = AutoModelForSequenceClassification.from_pretrained(
    CYBERBERT_MODEL_PATH,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)

print("✅ CyberBERT loaded for fine-tuning.")
print("Number of labels:", cyberbert_model.config.num_labels)
print("Problem Type:", cyberbert_model.config.problem_type)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Cyberbullying_Project/CyberBERT_Pretrained
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ CyberBERT loaded for fine-tuning.
Number of labels: 13
Problem Type: multi_label_classification


In [ ]:
# ==========================================
# Check CyberBERT Pretrained Configuration
# ==========================================

import json
import os

config_path = os.path.join(
    PROJECT_PATH,
    "CyberBERT_Pretrained",
    "config.json"
)

with open(config_path, "r") as f:
    config = json.load(f)

print("=" * 70)
print("CyberBERT configuration")
print("=" * 70)

for key in [
    "_name_or_path",
    "model_type",
    "architectures",
    "hidden_size",
    "num_hidden_layers",
    "num_attention_heads",
    "vocab_size"
]:
    print(f"{key}: {config.get(key)}")

CyberBERT configuration
_name_or_path: None
model_type: bert
architectures: ['BertModel']
hidden_size: 768
num_hidden_layers: 12
num_attention_heads: 12
vocab_size: 30522


In [ ]:
# ==========================================
# Step 14: Define Common Training Configuration
# ==========================================

import torch
from transformers import TrainingArguments

# Original training configuration
NUM_EPOCHS = 2
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# Seed will be specified for each individual run later
SEED = 42

print("Training configuration:")
print("Epochs              :", NUM_EPOCHS)
print("Train batch size    :", TRAIN_BATCH_SIZE)
print("Eval batch size     :", EVAL_BATCH_SIZE)
print("Learning rate       :", LEARNING_RATE)
print("Weight decay        :", WEIGHT_DECAY)
print("Random seed         :", SEED)
print("FP16 enabled        :", torch.cuda.is_available())

Training configuration:
Epochs              : 2
Train batch size    : 16
Eval batch size     : 16
Learning rate       : 2e-05
Weight decay        : 0.01
Random seed         : 42
FP16 enabled        : True


In [ ]:
# ==========================================
# Step 15: Define Evaluation Metrics
# ==========================================

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss
)


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    # Convert logits to probabilities
    probabilities = 1 / (1 + np.exp(-logits))

    # Apply threshold
    predictions = (probabilities >= 0.5).astype(int)

    # Exact Match Accuracy
    subset_accuracy = accuracy_score(
        labels,
        predictions
    )

    # Hamming Loss
    ham_loss = hamming_loss(
        labels,
        predictions
    )

    # Hamming Accuracy
    ham_accuracy = 1 - ham_loss

    # Micro metrics
    micro_precision = precision_score(
        labels,
        predictions,
        average="micro",
        zero_division=0
    )

    micro_recall = recall_score(
        labels,
        predictions,
        average="micro",
        zero_division=0
    )

    micro_f1 = f1_score(
        labels,
        predictions,
        average="micro",
        zero_division=0
    )

    # Macro metrics
    macro_precision = precision_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    # Weighted F1
    weighted_f1 = f1_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "Subset_Accuracy": subset_accuracy,
        "Hamming_Accuracy": ham_accuracy,
        "Hamming_Loss": ham_loss,
        "Micro_Precision": micro_precision,
        "Micro_Recall": micro_recall,
        "Micro_F1": micro_f1,
        "Macro_Precision": macro_precision,
        "Macro_Recall": macro_recall,
        "Macro_F1": macro_f1,
        "Weighted_F1": weighted_f1
    }


print("✅ Evaluation metrics defined successfully!")

✅ Evaluation metrics defined successfully!


In [ ]:
# ==========================================
# Cell 16: Fine-tune HateBERT with 3 Seeds
# ==========================================

import os
import shutil
import torch

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)


# ============================================================
# Three independent random seeds
# ============================================================

HATEBERT_SEEDS = [42, 123, 456]


# ============================================================
# Paths
# ============================================================

# Pretrained HateBERT model
HATEBERT_PRETRAINED_PATH = os.path.join(
    PROJECT_PATH,
    "HateBERT_Pretrained"
)

# Temporary training checkpoints
HATEBERT_CHECKPOINT_ROOT = os.path.join(
    PROJECT_PATH,
    "HateBERT_Seed_Checkpoints"
)

# Final seed-specific models
HATEBERT_SEED_MODEL_ROOT = os.path.join(
    PROJECT_PATH,
    "HateBERT_Seed_Models"
)

os.makedirs(
    HATEBERT_CHECKPOINT_ROOT,
    exist_ok=True
)

os.makedirs(
    HATEBERT_SEED_MODEL_ROOT,
    exist_ok=True
)


# ============================================================
# Start experiment
# ============================================================

print("=" * 70)
print("HATEBERT 3-SEED FINE-TUNING")
print("=" * 70)

print(f"Seeds: {HATEBERT_SEEDS}")
print(
    f"Training samples: "
    f"{len(hatebert_train_dataset)}"
)
print(
    f"Validation samples: "
    f"{len(hatebert_val_dataset)}"
)
print(f"Number of labels: {NUM_LABELS}")

print("=" * 70)


# ============================================================
# Train each seed independently
# ============================================================

for seed in HATEBERT_SEEDS:

    print("\n" + "=" * 70)
    print(f"Starting HateBERT training — Seed {seed}")
    print("=" * 70)

    # --------------------------------------------------------
    # Set random seed
    # --------------------------------------------------------

    set_seed(seed)

    # --------------------------------------------------------
    # Seed-specific directories
    # --------------------------------------------------------

    seed_checkpoint_path = os.path.join(
        HATEBERT_CHECKPOINT_ROOT,
        f"seed_{seed}"
    )

    seed_model_path = os.path.join(
        HATEBERT_SEED_MODEL_ROOT,
        f"seed_{seed}"
    )

    # --------------------------------------------------------
    # Remove old incomplete results if they exist
    # --------------------------------------------------------

    if os.path.exists(seed_checkpoint_path):
        shutil.rmtree(seed_checkpoint_path)

    if os.path.exists(seed_model_path):
        shutil.rmtree(seed_model_path)

    os.makedirs(
        seed_checkpoint_path,
        exist_ok=True
    )

    os.makedirs(
        seed_model_path,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Load a FRESH pretrained HateBERT model
    # --------------------------------------------------------

    hatebert_model = AutoModelForSequenceClassification.from_pretrained(
        HATEBERT_PRETRAINED_PATH,
        num_labels=NUM_LABELS,
        problem_type="multi_label_classification"
    )

    # --------------------------------------------------------
    # Training arguments
    # --------------------------------------------------------

    seed_training_args = TrainingArguments(

        output_dir=seed_checkpoint_path,

        # Training
        num_train_epochs=2,

        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,

        learning_rate=2e-5,
        weight_decay=0.01,

        # Logging
        logging_dir=os.path.join(
            seed_checkpoint_path,
            "logs"
        ),

        logging_strategy="steps",
        logging_steps=500,

        # Validation
        eval_strategy="steps",
        eval_steps=5000,

        # Checkpoint saving
        save_strategy="steps",
        save_steps=5000,

        save_total_limit=1,

        # Select best validation-loss checkpoint
        load_best_model_at_end=True,

        metric_for_best_model="eval_loss",
        greater_is_better=False,

        # GPU
        fp16=torch.cuda.is_available(),

        # Disable external logging
        report_to="none",

        # Reproducibility
        seed=seed,
        data_seed=seed
    )

    # --------------------------------------------------------
    # Create Trainer
    # --------------------------------------------------------

    hatebert_trainer = Trainer(

        model=hatebert_model,

        args=seed_training_args,

        train_dataset=hatebert_train_dataset,
        eval_dataset=hatebert_val_dataset,

        processing_class=HATEBERT_TOKENIZER,

        compute_metrics=compute_metrics
    )

    # --------------------------------------------------------
    # Start training
    # --------------------------------------------------------

    print(
        f"\n🚀 Training HateBERT with seed {seed}..."
    )

    hatebert_trainer.train()

    print(
        f"\n✅ HateBERT seed {seed} training completed."
    )

    # --------------------------------------------------------
    # Save best validation model
    # --------------------------------------------------------

    hatebert_trainer.save_model(
        seed_model_path
    )

    HATEBERT_TOKENIZER.save_pretrained(
        seed_model_path
    )

    print(
        "\n✅ Best HateBERT model saved:"
    )

    print(seed_model_path)

    # --------------------------------------------------------
    # Delete temporary checkpoints
    # --------------------------------------------------------

    if os.path.exists(seed_checkpoint_path):

        shutil.rmtree(
            seed_checkpoint_path
        )

        print(
            "🗑️ Temporary checkpoints deleted."
        )

    # --------------------------------------------------------
    # Clear memory before next seed
    # --------------------------------------------------------

    del hatebert_trainer
    del hatebert_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(
        f"✅ GPU memory cleared after seed {seed}."
    )


# ============================================================
# Experiment completed
# ============================================================

print("\n" + "=" * 70)
print("✅ ALL 3 HATEBERT SEED RUNS COMPLETED")
print("=" * 70)


# ============================================================
# Verify saved models
# ============================================================

print("\nSaved HateBERT seed models:")

for seed in HATEBERT_SEEDS:

    seed_path = os.path.join(
        HATEBERT_SEED_MODEL_ROOT,
        f"seed_{seed}"
    )

    print(
        f"• Seed {seed}: {seed_path}"
    )

print("\n" + "=" * 70)
print("HateBERT multi-seed experiment finished.")
print("=" * 70)

HATEBERT 3-SEED FINE-TUNING
Seeds: [42, 123, 456]
Training samples: 200000
Validation samples: 68439
Number of labels: 13

Starting HateBERT training — Seed 42


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Cyberbullying_Project/HateBERT_Pretrained
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



🚀 Training HateBERT with seed 42...


Step,Training Loss,Validation Loss,Subset Accuracy,Hamming Accuracy,Hamming Loss,Micro Precision,Micro Recall,Micro F1,Macro Precision,Macro Recall,Macro F1,Weighted F1
5000,0.156138,0.149537,0.576908,0.949733,0.050267,0.842735,0.561014,0.673605,0.834953,0.550948,0.642591,0.650307
10000,0.147695,0.141181,0.594310,0.951671,0.048329,0.850551,0.579018,0.688997,0.843448,0.567612,0.661373,0.669027
15000,0.127961,0.141530,0.607914,0.951507,0.048493,0.831579,0.596268,0.694534,0.821900,0.587638,0.670740,0.678015
20000,0.130364,0.138941,0.607724,0.952261,0.047739,0.843160,0.594189,0.697112,0.831680,0.584683,0.671493,0.679051
25000,0.126079,0.138242,0.609930,0.952485,0.047515,0.843475,0.596851,0.699049,0.831561,0.587324,0.674599,0.681931


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ HateBERT seed 456 training completed.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Best HateBERT model saved:
/content/drive/MyDrive/Cyberbullying_Project/HateBERT_Seed_Models/seed_456
🗑️ Temporary checkpoints deleted.
✅ GPU memory cleared after seed 456.

✅ ALL 3 HATEBERT SEED RUNS COMPLETED

Saved HateBERT seed models:
• Seed 42: /content/drive/MyDrive/Cyberbullying_Project/HateBERT_Seed_Models/seed_42
• Seed 123: /content/drive/MyDrive/Cyberbullying_Project/HateBERT_Seed_Models/seed_123
• Seed 456: /content/drive/MyDrive/Cyberbullying_Project/HateBERT_Seed_Models/seed_456

HateBERT multi-seed experiment finished.


In [ ]:
# ==========================================
# Cell 17: Fine-tune CyberBERT with 3 Seeds
# ==========================================

import os
import shutil
import torch

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)


# ============================================================
# Three independent random seeds
# ============================================================

CYBERBERT_SEEDS = [42, 123, 456]


# ============================================================
# Paths
# ============================================================

# Pretrained CyberBERT model
CYBERBERT_PRETRAINED_PATH = os.path.join(
    PROJECT_PATH,
    "CyberBERT_Pretrained"
)

# Temporary training checkpoints
CYBERBERT_CHECKPOINT_ROOT = os.path.join(
    PROJECT_PATH,
    "CyberBERT_Seed_Checkpoints"
)

# Final seed-specific models
CYBERBERT_SEED_MODEL_ROOT = os.path.join(
    PROJECT_PATH,
    "CyberBERT_Seed_Models"
)

os.makedirs(
    CYBERBERT_CHECKPOINT_ROOT,
    exist_ok=True
)

os.makedirs(
    CYBERBERT_SEED_MODEL_ROOT,
    exist_ok=True
)


# ============================================================
# Start experiment
# ============================================================

print("=" * 70)
print("CYBERBERT 3-SEED FINE-TUNING")
print("=" * 70)

print(f"Seeds: {CYBERBERT_SEEDS}")

print(
    f"Training samples: "
    f"{len(cyberbert_train_dataset)}"
)

print(
    f"Validation samples: "
    f"{len(cyberbert_val_dataset)}"
)

print(
    f"Number of labels: "
    f"{NUM_LABELS}"
)

print("=" * 70)


# ============================================================
# Train each seed independently
# ============================================================

for seed in CYBERBERT_SEEDS:

    print("\n" + "=" * 70)
    print(
        f"Starting CyberBERT training — Seed {seed}"
    )
    print("=" * 70)


    # --------------------------------------------------------
    # Set random seed
    # --------------------------------------------------------

    set_seed(seed)


    # --------------------------------------------------------
    # Seed-specific directories
    # --------------------------------------------------------

    seed_checkpoint_path = os.path.join(
        CYBERBERT_CHECKPOINT_ROOT,
        f"seed_{seed}"
    )

    seed_model_path = os.path.join(
        CYBERBERT_SEED_MODEL_ROOT,
        f"seed_{seed}"
    )


    # --------------------------------------------------------
    # Remove old incomplete results if they exist
    # --------------------------------------------------------

    if os.path.exists(seed_checkpoint_path):
        shutil.rmtree(seed_checkpoint_path)

    if os.path.exists(seed_model_path):
        shutil.rmtree(seed_model_path)


    # --------------------------------------------------------
    # Create directories
    # --------------------------------------------------------

    os.makedirs(
        seed_checkpoint_path,
        exist_ok=True
    )

    os.makedirs(
        seed_model_path,
        exist_ok=True
    )


    # --------------------------------------------------------
    # Load a FRESH pretrained CyberBERT model
    # --------------------------------------------------------

    cyberbert_model = (
        AutoModelForSequenceClassification.from_pretrained(
            CYBERBERT_PRETRAINED_PATH,
            num_labels=NUM_LABELS,
            problem_type="multi_label_classification"
        )
    )


    # --------------------------------------------------------
    # Training arguments
    # --------------------------------------------------------

    seed_training_args = TrainingArguments(

        output_dir=seed_checkpoint_path,

        # Training
        num_train_epochs=2,

        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,

        learning_rate=2e-5,
        weight_decay=0.01,

        # Logging
        logging_dir=os.path.join(
            seed_checkpoint_path,
            "logs"
        ),

        logging_strategy="steps",
        logging_steps=500,

        # Validation
        eval_strategy="steps",
        eval_steps=5000,

        # Checkpoint saving
        save_strategy="steps",
        save_steps=5000,

        save_total_limit=1,

        # Select best validation-loss checkpoint
        load_best_model_at_end=True,

        metric_for_best_model="eval_loss",
        greater_is_better=False,

        # GPU
        fp16=torch.cuda.is_available(),

        # Disable external logging
        report_to="none",

        # Reproducibility
        seed=seed,
        data_seed=seed
    )


    # --------------------------------------------------------
    # Create Trainer
    # --------------------------------------------------------

    cyberbert_trainer = Trainer(

        model=cyberbert_model,

        args=seed_training_args,

        # IMPORTANT:
        # Use CyberBERT-specific datasets
        train_dataset=cyberbert_train_dataset,
        eval_dataset=cyberbert_val_dataset,

        processing_class=CYBERBERT_TOKENIZER,

        compute_metrics=compute_metrics
    )


    # --------------------------------------------------------
    # Start training
    # --------------------------------------------------------

    print(
        f"\n🚀 Training CyberBERT with seed {seed}..."
    )

    cyberbert_trainer.train()

    print(
        f"\n✅ CyberBERT seed {seed} training completed."
    )


    # --------------------------------------------------------
    # Save best validation model
    # --------------------------------------------------------

    cyberbert_trainer.save_model(
        seed_model_path
    )

    CYBERBERT_TOKENIZER.save_pretrained(
        seed_model_path
    )

    print(
        "\n✅ Best CyberBERT model saved:"
    )

    print(seed_model_path)


    # --------------------------------------------------------
    # Delete temporary checkpoints
    # --------------------------------------------------------

    if os.path.exists(seed_checkpoint_path):

        shutil.rmtree(
            seed_checkpoint_path
        )

        print(
            "🗑️ Temporary checkpoints deleted."
        )


    # --------------------------------------------------------
    # Clear memory before next seed
    # --------------------------------------------------------

    del cyberbert_trainer
    del cyberbert_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(
        f"✅ GPU memory cleared after seed {seed}."
    )


# ============================================================
# Experiment completed
# ============================================================

print("\n" + "=" * 70)
print("✅ ALL 3 CYBERBERT SEED RUNS COMPLETED")
print("=" * 70)


# ============================================================
# Verify saved models
# ============================================================

print("\nSaved CyberBERT seed models:")

for seed in CYBERBERT_SEEDS:

    seed_path = os.path.join(
        CYBERBERT_SEED_MODEL_ROOT,
        f"seed_{seed}"
    )

    print(
        f"• Seed {seed}: {seed_path}"
    )


print("\n" + "=" * 70)
print("CyberBERT multi-seed experiment finished.")
print("=" * 70)

CYBERBERT 3-SEED FINE-TUNING
Seeds: [42, 123, 456]
Training samples: 200000
Validation samples: 68439
Number of labels: 13

Starting CyberBERT training — Seed 42


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Cyberbullying_Project/CyberBERT_Pretrained
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



🚀 Training CyberBERT with seed 42...


Step,Training Loss,Validation Loss


...

[transformers] BertForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Cyberbullying_Project/CyberBERT_Pretrained
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



🚀 Training CyberBERT with seed 123...


Step,Training Loss,Validation Loss,Subset Accuracy,Hamming Accuracy,Hamming Loss,Micro Precision,Micro Recall,Micro F1,Macro Precision,Macro Recall,Macro F1,Weighted F1
5000,0.155313,0.150577,0.580196,0.949256,0.050744,0.830923,0.566424,0.673640,0.826697,0.556506,0.640607,0.648658
10000,0.146347,0.145353,0.585865,0.950182,0.049818,0.834630,0.575140,0.681003,0.828588,0.570629,0.658326,0.664313


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]